# Feature Engineering & Unsupervised Learning

Raw data arrives in formats that most learning algorithms cannot consume directly. A continuous regressor expects a float matrix; a kernel SVM expects numeric inputs with finite norm; gradient-boosted trees tolerate missing values but not string categories. **Feature engineering** is the craft of transforming raw columns into numeric representations that expose the predictive signal a model can exploit. It is unglamorous work, but in practice it determines more of the final model quality than architecture choice does.

This notebook covers the four classical pillars of feature engineering and unsupervised learning: (1) encoding categorical variables, (2) selecting and reducing features, (3) finding low-dimensional structure with PCA, and (4) discovering latent groupings with clustering. We close with anomaly detection, which sits at the intersection of unsupervised learning and applied decision-making.

Imports used throughout the notebook:

In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")

## Encoding Categorical Features

Most tabular datasets contain columns whose values are categories — strings or integers that denote membership in a set rather than a position on a number line. A model that receives the raw integer code `{0, 1, 2}` for the cities `{"Austin", "Boston", "Chicago"}` will silently treat Chicago as twice Boston, which is nonsense. Choosing the right encoding is therefore not a preprocessing detail; it shapes the inductive bias baked into every downstream model.

**Ordinal encoding** replaces each category with an integer $0, 1, \ldots, K-1$ according to a specified order. It is correct when the order is meaningful — e.g. `{"cold", "warm", "hot"}` → `{0, 1, 2}` — and incorrect when applied to nominal categories. Tree-based models can recover from an arbitrary ordinal assignment by learning the right splits, but linear models and distance-based algorithms cannot.

**One-hot encoding** (OHE) maps each category to a binary indicator vector of length $K$. For a feature with $K$ categories, each sample becomes a row of the $K$-dimensional binary matrix $\mathbf{E} \in \{0, 1\}^{N \times K}$ where exactly one entry per row is $1$. OHE is correct for nominal categories and is the standard choice for linear models. Its weakness is **high cardinality**: a `user_id` column with $10^5$ distinct values produces a $10^5$-dimensional sparse matrix that overwhelms the model's feature space and makes storage expensive.

**Target encoding** replaces each category $c$ with the conditional mean of the target:

$$\hat{\mu}_c = \mathbb{E}[y \mid x = c] \approx \frac{1}{|\mathcal{D}_c|} \sum_{i \in \mathcal{D}_c} y_i,$$

where $\mathcal{D}_c = \{i : x_i = c\}$ is the set of training examples belonging to category $c$. This compresses any cardinality down to a single float and directly encodes the signal the model cares about. The danger is **target leakage**: if the encoding is computed on the full training set and then used as input to a model trained on the same set, the model sees the target through a side channel, inflating in-sample metrics and degrading out-of-sample performance. The remedy is cross-validated target encoding: split the training data into $K$ folds, compute the mean on the out-of-fold samples for each fold, and use those out-of-fold estimates as features. `sklearn.preprocessing.TargetEncoder` does this automatically.

**Hash encoding** (also called the **hashing trick**) maps each category to one of $B$ buckets by applying a hash function $h(c) \mod B$. It handles unlimited cardinality with no vocabulary to store and is trivially parallelizable, but it introduces **collision artifacts** — two distinct categories that hash to the same bucket are indistinguishable — and offers no interpretability.

**Embeddings** — learned dense representations of categorical values — are covered in the tabular deep learning notebook (NB06 of this series).

**When to use what.** Ordinal encoding for ordered categories with tree models. One-hot for nominal categories with low cardinality ($K \lesssim 50$) and linear or distance-based models. Target encoding for high-cardinality columns when you have enough data per category (at least $\sim$20 observations) and use cross-validation correctly. Hash encoding when cardinality is unbounded and speed matters more than interpretability.

**Example.** A synthetic dataset with a nominal and an ordinal feature:

In [ ]:
rng = np.random.default_rng(42)

cities = ["Austin", "Boston", "Chicago", "Denver"]
temps  = ["cold", "warm", "hot"]
N = 12

df = pd.DataFrame({
    "city":    rng.choice(cities, N),
    "temp":    rng.choice(temps, N),
    "revenue": rng.integers(10, 100, N).astype(float),
})
df.head(6)

Ordinal encoding for `temp` (preserves order) and one-hot for `city` (nominal):

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Ordinal — explicit ordering matters
ord_enc = OrdinalEncoder(categories=[["cold", "warm", "hot"]])
df["temp_ord"] = ord_enc.fit_transform(df[["temp"]])

# One-hot — nominal, no ordering
ohe = OneHotEncoder(sparse_output=False)
city_ohe = ohe.fit_transform(df[["city"]])
ohe_df = pd.DataFrame(city_ohe, columns=ohe.get_feature_names_out())

result = pd.concat([df[["city", "temp", "temp_ord"]], ohe_df], axis=1)
result.head(6)

**Target encoding leakage.** To see the leakage risk concretely, we encode on the full training set and measure in-sample vs. out-of-sample error:

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Naive: encode on the whole training set, then cross-validate
city_means = df.groupby("city")["revenue"].transform("mean")  # <1>
X_naive = city_means.values.reshape(-1, 1)
y = df["revenue"].values

model = Ridge()
r2_naive = cross_val_score(model, X_naive, y, cv=4, scoring="r2").mean()
print(f"Naive target encoding  R² = {r2_naive:.3f}")

1. `transform("mean")` broadcasts the per-group mean back to every row — including rows whose label contributed to that mean. This is the leakage.

Now the correct approach using `TargetEncoder` inside a pipeline — cross-validated encoding is done automatically on the held-out fold during `cross_val_score`:

In [ ]:
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline

# Correct: TargetEncoder inside a pipeline with cross-validation
pipe = Pipeline([
    ("enc", TargetEncoder(target_type="continuous", cv=4)),  # <1>
    ("reg", Ridge()),
])

X_city = df[["city"]]
r2_correct = cross_val_score(pipe, X_city, y, cv=4, scoring="r2").mean()
print(f"Pipeline target encoding R² = {r2_correct:.3f}")

1. `TargetEncoder` with `cv=4` computes out-of-fold means during `fit_transform`, so no label leaks into the encoding of any sample.

:::{.callout-caution}
Computing target encoding statistics on the full training set and then using that encoding as input during cross-validation inflates $R^2$ (or any metric) significantly. Always wrap `TargetEncoder` in a `Pipeline` so that `cross_val_score` or `GridSearchCV` refits the encoder on each training fold.

:::

## Feature Selection

Adding features costs nothing during data collection but imposes costs during training: more parameters, longer training, and — critically — the **curse of dimensionality**. As $d$ grows, the volume of the feature space grows as $\mathcal{O}(r^d)$ for a ball of radius $r$, so any fixed training set $\mathcal{D}$ becomes exponentially sparse. Distances lose discriminative power; nearest neighbors become nearly equidistant; and models require exponentially more data to maintain the same statistical coverage. Feature selection combats this by retaining only the $k \ll d$ features that carry the most signal.

**Filter methods** score each feature independently of the model and select the top-$k$. Pearson correlation measures linear association with the target: fast and interpretable, but blind to nonlinear relationships. **Mutual information** $I(X_j; Y) = \mathbb{E}\left[\log \frac{p(X_j, Y)}{p(X_j)p(Y)}\right]$ captures any statistical dependence, linear or not. It is the preferred filter for tree and kernel models. Both methods are model-agnostic and scale to millions of samples.

**Wrapper methods** treat feature subsets as a search problem. **Recursive Feature Elimination** (RFE) fits the model, ranks features by importance, removes the weakest, and repeats until $k$ features remain. It is model-aware and can capture interactions, but its cost is $\mathcal{O}(d)$ model fits — prohibitive for large $d$ or slow models.

**Embedded methods** perform selection during training. $L_1$ (Lasso) regularization drives irrelevant feature weights to exactly zero, yielding a sparse solution from a single fit. Tree ensembles accumulate an **impurity-based importance** for each feature equal to the total reduction in the Gini impurity (or variance, for regression) attributable to splits on that feature. These importances are fast and cheap but can be misleading for high-cardinality or correlated features; **permutation importances** are a more reliable alternative.

**Practical rule.** For a first pass, use mutual information as a filter to cut obvious dead weight. If computational budget allows, follow with RFE or embedded importances to account for feature interactions.

**Data.** The digits dataset from scikit-learn: $N = 1797$ images of hand-written digits, each flattened to $d = 64$ pixel features.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X, y = digits.data, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

Selecting the top-$k$ features by mutual information with `SelectKBest`:

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

results = {}
for k in [8, 16, 32, 64]:
    pipe = Pipeline([
        ("sel", SelectKBest(mutual_info_classif, k=k)),  # <1>
        ("clf", LogisticRegression(max_iter=1000)),
    ])
    pipe.fit(X_tr, y_tr)
    results[k] = pipe.score(X_te, y_te)

for k, acc in results.items():
    print(f"  k={k:2d}  test accuracy = {acc:.3f}")

1. `mutual_info_classif` estimates $I(X_j; Y)$ via $k$-nearest-neighbor density estimation — it handles non-linear dependencies automatically.

**Result.** With only $k = 32$ out of $64$ features the accuracy is competitive with the full-feature model, confirming that roughly half the pixel positions carry little discriminative information (primarily the border pixels of an 8×8 image).

## Principal Component Analysis

Feature selection discards features entirely. **Principal Component Analysis** (PCA) takes a different approach: it constructs $r \ll d$ new features as linear combinations of the original ones, chosen to capture the directions of maximum variance in the data.

**Mechanics.** Center the data matrix $\mathbf{X} \in \mathbb{R}^{N \times d}$ by subtracting column means. The **thin SVD** of the centered matrix decomposes it as:

$$\mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^\top,$$

where $\mathbf{U} \in \mathbb{R}^{N \times d}$ has orthonormal columns, $\mathbf{\Sigma} \in \mathbb{R}^{d \times d}$ is diagonal with singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_d \geq 0$, and $\mathbf{V} \in \mathbb{R}^{d \times d}$ is orthonormal. The columns $\mathbf{v}_1, \ldots, \mathbf{v}_d$ of $\mathbf{V}$ are the **principal components** (PCs) — the eigenvectors of the sample covariance matrix $\frac{1}{N-1}\mathbf{X}^\top \mathbf{X}$. Projecting $\mathbf{X}$ onto the first $r$ PCs gives the low-dimensional representation:

$$\mathbf{Z} = \mathbf{X} \mathbf{V}_r \in \mathbb{R}^{N \times r},$$

where $\mathbf{V}_r$ contains the first $r$ columns of $\mathbf{V}.$

The **explained variance ratio** of component $j$ is $\sigma_j^2 / \sum_l \sigma_l^2$, measuring the fraction of total variance captured. Plotting these ratios — the **scree plot** — reveals the intrinsic dimensionality of the data as the elbow where additional components contribute diminishing returns.

**When PCA helps.** PCA benefits linear models (it decorrelates features and removes multicollinearity), compressed sensing tasks, and visualization ($r = 2$ or $3$). It is less useful for tree ensembles, which are invariant to linear transformations of the features, and harmful for sparse text features, where the dense PCA projection destroys the sparsity that makes those features efficient.

Fitting PCA on the digits dataset and inspecting explained variance:

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X)  # <1>
evr = pca.explained_variance_ratio_
cumulative = np.cumsum(evr)

n90 = np.searchsorted(cumulative, 0.90) + 1  # <2>
n95 = np.searchsorted(cumulative, 0.95) + 1
print(f"Components for 90% variance: {n90}")
print(f"Components for 95% variance: {n95}")

1. Fitting with no `n_components` argument computes all $\min(N, d)$ components — useful for drawing the full scree plot before committing to $r$.
2. `searchsorted` returns the first index where the cumulative sum exceeds the threshold; adding 1 converts to 1-indexed component count.

In [ ]:
#| label: fig-pca-variance
#| fig-cap: "Scree plot of explained variance ratio (bar) and cumulative explained variance (line) for the principal components of the digits dataset ($d = 64$). The 90% and 95% thresholds are marked with dashed horizontals."
#| code-fold: true

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

digits = load_digits()
pca_plot = PCA().fit(digits.data)
evr = pca_plot.explained_variance_ratio_
cumulative = np.cumsum(evr)
n_components = len(evr)
idx = np.arange(1, n_components + 1)

n90 = np.searchsorted(cumulative, 0.90) + 1
n95 = np.searchsorted(cumulative, 0.95) + 1

fig, ax = plt.subplots(figsize=(8, 4))

# Bar chart for individual explained variance
ax.bar(idx, evr, color="C0", alpha=0.6, label="Explained variance ratio")

# Cumulative line on same axis
ax.plot(idx, cumulative, color="C1", linewidth=2, label="Cumulative")

# Threshold lines
ax.axhline(0.90, color="gray", linestyle="dashed", lw=1.0, label="90%")
ax.axhline(0.95, color="gray", linestyle="dotted", lw=1.0, label="95%")

# Annotate component counts
ax.annotate(
    f"PC {n90}", xy=(n90, 0.90), xytext=(n90 + 2, 0.84),
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
    fontsize=8, color="gray"
)
ax.annotate(
    f"PC {n95}", xy=(n95, 0.95), xytext=(n95 + 2, 0.89),
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
    fontsize=8, color="gray"
)

ax.set_xlabel("Principal component"); ax.set_ylabel("Explained variance ratio")
ax.set_xlim(0.5, n_components + 0.5); ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, loc="center right")
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

**Figure.** The scree plot shows that the variance concentrates sharply in the first few components. The cumulative curve reaches 90% around component 21 and 95% around component 29, out of 64 total — confirming that the 64-pixel space has substantial redundancy.

Comparing a logistic regression trained on the full feature set vs. PCA-compressed representations:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

configs = {"Full (d=64)": None, "PCA r=21": 21, "PCA r=29": 29, "PCA r=10": 10}
for name, n_comp in configs.items():
    steps = []
    if n_comp is not None:
        steps.append(("pca", PCA(n_components=n_comp)))
    steps.append(("clf", LogisticRegression(max_iter=1000)))
    pipe = Pipeline(steps)
    pipe.fit(X_tr, y_tr)
    acc = pipe.score(X_te, y_te)
    print(f"  {name:<18}  test accuracy = {acc:.3f}")

**Result.** PCA at 95% variance retention ($r = 29$) matches the full-feature accuracy almost exactly, while $r = 21$ (90%) incurs a small drop. Compressing aggressively to $r = 10$ loses meaningful signal. This confirms the standard heuristic: choose $r$ to retain 90–95% of variance as a starting point, then tune.

## Clustering: K-Means and HDBSCAN

Clustering partitions unlabeled data into groups of similar points without supervision. It is used for exploratory analysis, customer segmentation, anomaly pre-screening, and as a preprocessing step that creates categorical features from continuous spaces.

**K-Means** minimizes the within-cluster sum of squared distances:

$$J = \sum_{j=1}^k \sum_{i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2,$$

where $C_j$ is the set of points assigned to cluster $j$ and $\boldsymbol{\mu}_j$ is the cluster centroid. **Lloyd's algorithm** alternates between two steps: (1) assign each point to its nearest centroid (Voronoi partition), and (2) recompute each centroid as the mean of its assigned points. The algorithm converges to a local minimum, so initialization matters. **K-Means++** selects initial centroids with probability proportional to $d^2$ — the squared distance from the nearest existing centroid — reducing the chance of poor convergence.

K-Means makes three strong assumptions: clusters are (1) convex, (2) roughly equal-sized, and (3) approximately isotropic. On elongated, crescent-shaped, or nested clusters it fails structurally — not because $k$ was chosen wrong, but because the objective function cannot represent those shapes.

The **elbow method** plots $J$ as a function of $k$ and looks for a "knee" where adding another cluster yields diminishing returns. It works when well-separated blob clusters exist but is unreliable for complex geometries.

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise) takes a fundamentally different approach: it identifies regions of high point density and treats low-density regions as cluster boundaries. Its key properties are: (1) it discovers $k$ automatically rather than requiring it as input, (2) it handles arbitrary cluster shapes, (3) it assigns a special label of $-1$ to **noise points** that do not belong to any dense region, and (4) it is robust to choice of the `min_cluster_size` hyperparameter. `sklearn.cluster.HDBSCAN` is available in scikit-learn 1.3+.

In [ ]:
#| label: fig-cluster-comparison
#| fig-cap: "K-Means vs. HDBSCAN on three synthetic 2D datasets. K-Means fails on non-convex geometries (moons and circles) because it partitions space via Voronoi cells; HDBSCAN recovers the true structure and correctly marks outliers as noise (grey)."
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
SEED = 0

# Datasets
datasets = [
    ("Moons",   *make_moons(n_samples=300, noise=0.08, random_state=SEED)),
    ("Circles", *make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=SEED)),
    ("Blobs",   *make_blobs(n_samples=300, centers=3, cluster_std=0.7, random_state=SEED)),
]

# Methods: (label, constructor)
methods = [
    ("K-Means", lambda k: KMeans(n_clusters=k, n_init="auto", random_state=SEED)),
    ("HDBSCAN", lambda _:  HDBSCAN(min_cluster_size=20)),
]

# K-Means k per dataset (known ground truth)
k_true = {"Moons": 2, "Circles": 2, "Blobs": 3}

# Colormap: noise (-1) → grey, clusters → tab10
cmap = plt.get_cmap("tab10")

def cluster_colors(labels):
    colors = []
    for lbl in labels:
        if lbl == -1:
            colors.append("lightgrey")
        else:
            colors.append(cmap(lbl % 10))
    return colors

fig, axes = plt.subplots(2, 3, figsize=(10, 6))

for col, (name, X_raw, y_true) in enumerate(datasets):
    X_scaled = StandardScaler().fit_transform(X_raw)
    for row, (method_name, constructor) in enumerate(methods):
        ax = axes[row, col]
        k = k_true[name]
        clf = constructor(k)
        labels = clf.fit_predict(X_scaled)
        ari = adjusted_rand_score(y_true, labels)
        colors = cluster_colors(labels)
        ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=colors, s=8, linewidths=0)
        ax.set_title(f"{method_name} — {name}\nARI = {ari:.2f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

fig.tight_layout()
plt.show();

**Figure.** The Adjusted Rand Index (ARI) measures agreement between predicted and true cluster assignments, corrected for chance ($\text{ARI} = 1$ is perfect, $0$ is random). K-Means achieves near-perfect ARI on the blobs but fails on moons and circles because those datasets violate its convexity assumption. HDBSCAN recovers all three structures cleanly, with noise points (grey) marking the sparse inter-cluster regions.

**Elbow method** for selecting $k$ in K-Means:

In [ ]:
#| code-fold: true

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np

X_blob, _ = make_blobs(n_samples=500, centers=4, cluster_std=0.9, random_state=1)

inertias = []
ks = range(1, 11)
for k in ks:
    km = KMeans(n_clusters=k, n_init="auto", random_state=0)
    km.fit(X_blob)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(list(ks), inertias, marker="o", color="C0", linewidth=2)
ax.axvline(4, color="gray", linestyle="dashed", lw=1.0, label="True k=4")
ax.set_xlabel("Number of clusters $k$")
ax.set_ylabel("Inertia (within-cluster SS)")
ax.legend(fontsize=8)
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

**Figure.** The inertia drops sharply from $k=1$ to $k=4$ and flattens afterward — the elbow is visible at $k=4$, matching the true number of blobs. On real data the elbow is often indistinct; silhouette score or gap statistic can provide a more principled alternative.

**Choosing between K-Means and HDBSCAN.** Use K-Means when: clusters are expected to be convex and roughly spherical; $k$ is known or can be determined by the elbow method; speed is critical (K-Means scales to millions of points). Use HDBSCAN when: cluster shapes are unknown or non-convex; detecting noise/outliers is part of the task; and you want to avoid specifying $k$ in advance.

## Anomaly Detection

Anomaly detection identifies samples that deviate significantly from the bulk of the data. Unlike supervised classification, it operates without labeled anomalies — or with only normal examples — making it an unsupervised problem with asymmetric class structure.

**Isolation Forest** exploits the observation that anomalies are rare and different: they are isolated by random axis-aligned splits in far fewer steps than normal points. The algorithm builds an ensemble of random trees, each constructed by recursively splitting on a randomly chosen feature at a randomly chosen threshold. The **anomaly score** for a point $\mathbf{x}$ is the average path length across trees, normalized so that shorter paths yield higher anomaly scores. It scales to high-dimensional data, is insensitive to feature scale, and handles multimodal distributions without assumptions.

**Local Outlier Factor** (LOF) computes a **local reachability density** $\text{lrd}(\mathbf{x})$ for each point and then compares it with the densities of its $k$ nearest neighbors. Points whose neighborhood is denser than their own local region receive a high LOF score:

$$\text{LOF}_k(\mathbf{x}) = \frac{\sum_{\mathbf{o} \in N_k(\mathbf{x})} \text{lrd}(\mathbf{o}) / \text{lrd}(\mathbf{x})}{|N_k(\mathbf{x})|}.$$

LOF is effective when anomalies are defined relative to local neighborhood density — e.g. a medium-density cluster embedded in a high-density region. It is more expensive than Isolation Forest ($\mathcal{O}(N^2)$ in the worst case) and should not be used for novelty detection on new points unless fitted accordingly.

**One-class SVM** fits a hypersphere (in kernel space) that contains most of the training data. New points outside the sphere are flagged as anomalies. It is effective for smooth, unimodal distributions but sensitive to hyperparameter tuning and does not scale beyond $\sim 10^4$ samples.

**Reconstruction error** from a trained autoencoder is a powerful anomaly signal when the model is trained only on normal data: normal inputs reconstruct well (small error) while anomalies do not. This approach is covered in the deep learning series.

**Threshold selection.** All detectors produce a real-valued score, not a binary label. Choosing a threshold is a business decision: if a missed anomaly costs $c_{\text{FN}}$ and a false alarm costs $c_{\text{FP}}$, the optimal threshold minimizes expected cost $c_{\text{FN}} \cdot \text{FNR} + c_{\text{FP}} \cdot \text{FPR}$. When labeled examples of anomalies exist, the precision-recall curve over score thresholds guides the choice; otherwise the `contamination` parameter (expected fraction of outliers) serves as a prior.

**Data.** A 2D Gaussian inlier distribution with a small fraction of injected uniform outliers:

In [ ]:
rng = np.random.default_rng(7)

N_in  = 300
N_out = 30

X_in  = rng.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], size=N_in)
X_out = rng.uniform(-5, 5, size=(N_out, 2))  # <1>
X_all = np.vstack([X_in, X_out])
y_true = np.array([1] * N_in + [-1] * N_out)  # sklearn convention: 1=inlier, -1=outlier

print(f"Inliers: {N_in}, Outliers: {N_out}, Contamination: {N_out/(N_in+N_out):.2%}")

1. Outliers are drawn uniformly over a wide box — many will appear in sparse regions far from the Gaussian core.

Fitting Isolation Forest and evaluating precision and recall:

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report

contamination = N_out / (N_in + N_out)

iforest = IsolationForest(contamination=contamination, random_state=0)
y_iforest = iforest.fit_predict(X_all)  # returns +1 (inlier) or -1 (outlier)

lof = LocalOutlierFactor(contamination=contamination)
y_lof = lof.fit_predict(X_all)

print("=== Isolation Forest ===")
print(classification_report(y_true, y_iforest, target_names=["outlier", "inlier"]))
print("=== Local Outlier Factor ===")
print(classification_report(y_true, y_lof, target_names=["outlier", "inlier"]))

Decision boundary of Isolation Forest over the 2D plane:

In [ ]:
#| code-fold: true

import matplotlib.pyplot as plt
import numpy as np

xx, yy = np.meshgrid(np.linspace(-6, 6, 300), np.linspace(-6, 6, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
scores = iforest.decision_function(grid).reshape(xx.shape)  # <1>

fig, ax = plt.subplots(figsize=(6, 5))

# Decision surface
contour = ax.contourf(xx, yy, scores, levels=20, cmap="RdYlGn", alpha=0.6)
ax.contour(xx, yy, scores, levels=[0], colors="k", linewidths=1.5)  # <2>
plt.colorbar(contour, ax=ax, label="Anomaly score")

# Data points
ax.scatter(X_in[:, 0],  X_in[:, 1],  c="C0", s=15, label="Inlier",  linewidths=0)
ax.scatter(X_out[:, 0], X_out[:, 1], c="red",  s=40, marker="x", lw=1.5, label="Outlier")

ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$")
ax.legend(fontsize=9, loc="upper right")
ax.grid(linestyle="dotted", alpha=0.4)
fig.tight_layout()
plt.show();

1. `decision_function` returns the raw anomaly score (higher = more normal); the decision boundary is at $0$.
2. The black contour at level $0$ separates the predicted inlier region (green) from the outlier region (red).

**Figure.** The Isolation Forest correctly encircles the Gaussian inlier cluster and assigns high anomaly scores to points in the sparse peripheral regions. A few uniform outliers that happen to land near the Gaussian core are missed — this is expected given that those points are indistinguishable from inliers by any density-based criterion.

---

■